<a href="https://colab.research.google.com/github/azrael56/Evaluacion-1-Deep-Learning/blob/main/Deep_Learning_EVA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Evaluación Parcial 2: Implementación y Optimización de Redes Convolucionales (CNN)
**Asignatura:** Deep Learning (DLY0100)  
**Proyecto:** Clasificación de Personajes de Los Simpsons  

## 1. Introducción
El objetivo de este informe y cuaderno de trabajo es diseñar, evaluar y optimizar una Red Neuronal Convolucional (CNN) capaz de clasificar imágenes de distintos personajes de la serie "The Simpsons".

A lo largo del proyecto, se aplicarán conceptos fundamentales de visión por computadora para dar cumplimiento a los siguientes hitos de la rúbrica institucional:
1. Definición y entrenamiento de una arquitectura CNN Base (*Baseline*).
2. Análisis de métricas de rendimiento iniciales (Loss y Accuracy).
3. Implementación de técnicas de optimización, ajuste de hiperparámetros y regularización.
4. Uso de arquitecturas avanzadas y Transfer Learning para la comparación de resultados.

---
## 2. Carga, Normalización y Preprocesamiento de Datos
En esta sección importamos las librerías principales de TensorFlow y Keras. Utilizamos la clase `ImageDataGenerator` para normalizar los valores de los píxeles mediante `rescale=1./255` y segmentar el dataset utilizando un `validation_split=0.2`.

In [5]:
# Conectar a Google Drive (Si ya lo ejecutaste y diste permisos, no pasa nada si se vuelve a correr)
from google.colab import drive
drive.mount('/content/drive')

# Importar librerías de Deep Learning y Visualización
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print("Versión de TensorFlow detectada:", tf.__version__)

# Ruta definitiva del dataset
DATASET_PATH = '/content/drive/MyDrive/Deeplearning/simpsons_dataset'

# Configuración del generador base de imágenes
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

print("\n--- Cargando datos de Entrenamiento ---")
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

print("\n--- Cargando datos de Validación ---")
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Versión de TensorFlow detectada: 2.20.0

--- Cargando datos de Entrenamiento ---
Found 33511 images belonging to 43 classes.

--- Cargando datos de Validación ---
Found 8355 images belonging to 43 classes.


## 3. Arquitectura del Modelo Base (Baseline)
Para establecer un punto de comparación inicial (Baseline) frente a futuras mejoras, replicamos la estructura secuencial del archivo starter proporcionado. Esta red inicial cuenta con convoluciones simples, max pooling, dropout al 50% y una capa densa de 128 neuronas.

In [6]:
# Definición de la Red Neuronal Convolucional Base
model_base = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_generator.num_classes, activation='softmax')
])

model_base.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_base.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,630,699 (6.22 MB)

 Trainable params: 1,630,699 (6.22 MB)

 Non-trainable params: 0 (0.00 B)

## 4. Entrenamiento y Evaluación del Modelo Base
Entrenaremos la red convolucional durante 10 épocas y graficaremos el comportamiento de las curvas de Pérdida (*Loss*) y Precisión (*Accuracy*) para diagnosticar el estado del modelo.

In [ ]:
# Entrenamiento
epochs_base = 10
history_base = model_base.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs_base
)

# Función reutilizable para graficar
def plot_history(history, title="Rendimiento del Modelo"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(14, 5))

    # Gráfico de la métrica Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy', color='blue', linewidth=2)
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='orange', linewidth=2)
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.title(f'{title} - Accuracy Global')

    # Gráfico de la métrica Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss', color='blue', linewidth=2)
    plt.plot(epochs_range, val_loss, label='Validation Loss', color='orange', linewidth=2)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.title(f'{title} - Función de Pérdida (Loss)')

    plt.show()

# Graficar y evaluar modelo base
plot_history(history_base, title="Modelo Base (Baseline)")
loss, acc = model_base.evaluate(val_generator)
print(f"Loss Validación: {loss:.4f} | Accuracy Validación: {acc:.4f}")

Epoch 1/10
   9/1048 ━━━━━━━━━━━━━━━━━━━━ 3:41:51 13s/step - accuracy: 0.3431 - loss: 3.1194

## 5. Fase de Optimización: Data Augmentation
Para combatir el sobreajuste evidenciado en el modelo base, implementaremos **Data Augmentation**. Esta técnica aplica transformaciones aleatorias (rotaciones, desplazamientos, zoom y espejado horizontal) a las imágenes de entrenamiento en tiempo real, mejorando la generalización.

In [ ]:
# Nueva configuración con Data Augmentation (Solo para entrenamiento)
train_datagen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

# Nuevo generador de entrenamiento con imágenes aumentadas
print("\n--- Cargando datos de Entrenamiento (Aumentados) ---")
train_generator_aug = train_datagen_aug.flow_from_directory(
    DATASET_PATH,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

## 6. Arquitectura del Modelo Optimizado y Callbacks
Proponemos una arquitectura CNN más profunda y regularizada:
* Añadimos una tercera capa `Conv2D` de 128 filtros.
* Implementamos `BatchNormalization` después de cada convolución para estabilizar y acelerar el entrenamiento.
* Incrementamos las neuronas densas a 256.

Además, integramos **Callbacks**:
* **EarlyStopping:** Detiene el entrenamiento si no hay mejora, previniendo el sobreajuste de épocas altas.
* **ReduceLROnPlateau:** Ajusta dinámicamente el Learning Rate si el modelo se estanca.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Definición de la Red Convolucional Optimizada
model_opt = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(64, 64, 3), padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_generator_aug.num_classes, activation='softmax')
])

model_opt.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Definición de Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-5, verbose=1)

# Entrenamiento del modelo optimizado (Le damos 30 épocas, EarlyStopping lo detendrá antes si es necesario)
epochs_opt = 30
history_opt = model_opt.fit(
    train_generator_aug,
    validation_data=val_generator,
    epochs=epochs_opt,
    callbacks=[early_stop, reduce_lr]
)

# Evaluación y visualización
plot_history(history_opt, title="Modelo Optimizado (Data Aug + BatchNorm)")
loss_opt, acc_opt = model_opt.evaluate(val_generator)
print(f"Loss Validación Opt: {loss_opt:.4f} | Accuracy Validación Opt: {acc_opt:.4f}")

## 7. Fase de Transfer Learning (VGG16)
Para dar cumplimiento al requerimiento de comparar arquitecturas, implementaremos **Transfer Learning** utilizando **VGG16**, un modelo pre-entrenado con el dataset ImageNet (que contiene millones de imágenes).

**¿Por qué hacemos esto?**
En lugar de entrenar una red desde cero para que aprenda a detectar bordes, colores y formas, tomamos el conocimiento que VGG16 ya tiene (congelando sus capas base) y solo entrenamos una nueva capa densa final (clasificador) específica para las clases de Los Simpsons. Esto suele resultar en una convergencia más rápida y una mayor precisión.

In [ ]:
from tensorflow.keras.applications import VGG16

print("\n--- Construyendo Modelo con Transfer Learning (VGG16) ---")

# 1. Importar el modelo base VGG16
# include_top=False significa que NO traemos la capa final original de 1000 clases,
# porque nosotros queremos clasificar solo a los personajes de Los Simpsons.
base_model_vgg = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(64, 64, 3)
)

# 2. Congelar el modelo base (Para que sus pesos no se modifiquen al inicio)
base_model_vgg.trainable = False

# 3. Construir el nuevo modelo secuencial uniendo VGG16 con nuestro clasificador
model_tl = models.Sequential([
    base_model_vgg,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(), # Normalizamos antes del dropout
    layers.Dropout(0.5),
    layers.Dense(train_generator_aug.num_classes, activation='softmax')
])

# 4. Compilar el modelo
model_tl.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Ver la estructura: Notarás que la mayoría de los parámetros son "Non-trainable" (congelados)
model_tl.summary()

## 8. Entrenamiento y Evaluación del Modelo VGG16
Entrenaremos este nuevo modelo utilizando los mismos generadores con *Data Augmentation* y los mismos *Callbacks* (Early Stopping y ReduceLROnPlateau) que usamos en la fase anterior para que la comparación sea lo más justa posible.

In [ ]:
# Entrenamos el modelo con Transfer Learning
epochs_tl = 30 # Early stopping lo detendrá cuando sea óptimo

history_tl = model_tl.fit(
    train_generator_aug,
    validation_data=val_generator,
    epochs=epochs_tl,
    callbacks=[early_stop, reduce_lr]
)

# Visualizar resultados usando nuestra función
plot_history(history_tl, title="Modelo Transfer Learning (VGG16)")

# Evaluar el modelo final
loss_tl, acc_tl = model_tl.evaluate(val_generator)
print(f"\n==========================================")
print(f" MÉTRICAS FINALES TRANSFER LEARNING (VGG16)")
print(f"==========================================")
print(f"Loss en Validación:      {loss_tl:.4f}")
print(f"Accuracy en Validación:  {acc_tl:.4f}")
print(f"==========================================")

## 9. Fase de Evaluación Final: Matriz de Confusión y Reporte
Para comprender profundamente el comportamiento de nuestro mejor modelo (Transfer Learning con VGG16), no basta con la métrica de Accuracy global.

Utilizaremos una **Matriz de Confusión** y un **Reporte de Clasificación** para evaluar el rendimiento clase por clase. Esto nos permite identificar si existe un desbalance en el aprendizaje o si la red tiene dificultades extrayendo características distintivas entre personajes visualmente similares.

*Nota: Para generar esta matriz correctamente, debemos re-instanciar el generador de validación con `shuffle=False` para que el orden de las predicciones coincida exactamente con las etiquetas reales.*

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. Re-instanciar validación SIN mezclar (shuffle=False)
val_generator_cm = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False # ¡CRUCIAL para la Matriz de Confusión!
)

# 2. Generar predicciones usando el mejor modelo (model_tl)
print("\nGenerando predicciones sobre el set de validación...")
Y_pred = model_tl.predict(val_generator_cm)
y_pred_classes = np.argmax(Y_pred, axis=1) # Tomar la clase con mayor probabilidad
y_true = val_generator_cm.classes

# Nombres de las clases (personajes)
class_names = list(val_generator_cm.class_indices.keys())

# 3. Imprimir el Reporte de Clasificación (Precision, Recall, F1-Score)
print("\n=======================================================")
print("             REPORTE DE CLASIFICACIÓN                  ")
print("=======================================================")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

# 4. Graficar la Matriz de Confusión
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusión - Modelo Transfer Learning (VGG16)', fontsize=16)
plt.ylabel('Etiqueta Real (Personaje Verdadero)', fontsize=12)
plt.xlabel('Predicción del Modelo', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 10. Conclusión y Selección de Arquitectura

A lo largo de este proyecto, se implementaron y compararon progresivamente distintas arquitecturas de redes neuronales convolucionales (CNN) para resolver el problema de clasificación de imágenes de los personajes de "The Simpsons".

La evolución del modelo comenzó con una arquitectura base sencilla, la cual sirvió como punto de partida (*baseline*). Sin embargo, como suele ocurrir con modelos iniciales sin regularización, esta red presentó limitaciones para generalizar, tendiendo al sobreajuste (*overfitting*) al memorizar los datos de entrenamiento.

Para mitigar este problema, se desarrolló un segundo modelo optimizado que integró técnicas fundamentales de Deep Learning: se aplicó *Data Augmentation* para enriquecer artificialmente el conjunto de datos y *Batch Normalization* junto con *Callbacks* (como *Early Stopping*) para estabilizar y optimizar los tiempos de entrenamiento. Estas mejoras permitieron robustecer la red significativamente.

Finalmente, la solución definitiva y el modelo seleccionado para este caso es el basado en **Transfer Learning utilizando la arquitectura pre-entrenada VGG16**. El impacto de esta elección radica en su capacidad superior para extraer características visuales complejas (como bordes, texturas y formas). Al aprovechar el conocimiento previo que esta red adquirió entrenando con millones de imágenes (ImageNet) y adaptando únicamente sus capas finales a nuestro problema específico, se logró reducir drásticamente el error de clasificación.

En conclusión, el uso de Transfer Learning no solo aceleró la convergencia del modelo, sino que dotó a la solución de una capacidad de generalización muy superior, resultando en una herramienta altamente confiable y precisa para la inferencia y clasificación de nuevas imágenes.